# Mini-GPT do zero: Transformer com self-attention, treinado em texto proprio

Este notebook implementa um modelo estilo GPT **completo** (tokenizacao, self-attention, multi-head attention, blocos residuais, treino e geracao de texto), seguindo o espirito minimalista do nanoGPT de Andrej Karpathy.

**Requer PyTorch** (`pip install torch`) — usamos aqui porque precisamos de autograd automatico para treinar uma rede com dezenas de milhares de parametros; refazer isso a mao (como fizemos na Fase 1 com o MLP) seria impraticavel neste tamanho de arquitetura.

**Corpus de demonstracao**: um texto curto original escrito para este notebook, so para o modelo ter algo para "aprender o estilo" rapidamente. **Troque pelo seu proprio corpus** (letras de um artista, um livro, suas proprias anotacoes) para resultados mais interessantes — apenas garanta que voce tem o direito de usar esse texto.

## 1. Corpus e tokenizacao (nivel de caractere)

In [ ]:
text = """no comeco havia apenas ruido.
um sinal fraco em meio ao silencio de um universo que ainda nao sabia contar historias.
entao, letra por letra, o padrao comecou a emergir.
primeiro os sons, depois as palavras, depois as frases que se repetiam como mares
batendo na mesma praia, noite apos noite, ate que o ritmo virou memoria.
e assim, de padrao em padrao, o universo aprendeu a se contar para si mesmo."""

# troque a linha acima por, por exemplo:
# with open("meu_corpus.txt", "r", encoding="utf-8") as f:
#     text = f.read()

print(f"Tamanho do corpus: {len(text)} caracteres")


Tokenizacao por caractere e a mais simples possivel: cada caractere unico vira um token. LLMs de producao usam BPE (subpalavras), mas por caractere e mais facil de entender e suficiente para este exercicio — veja a Fase 2, Semana 1, para BPE.

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulario: {vocab_size} caracteres unicos -> {''.join(chars)!r}")

stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

def encode(s):
    return [stoi[c] for c in s]

def decode(l):
    return "".join(itos[i] for i in l)

# sanity check: codificar e decodificar deve devolver o texto original
assert decode(encode(text)) == text
print("Tokenizacao ok. Exemplo:", encode(text[:15]))


## 2. Preparar dados de treino

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Treino: {len(train_data)} tokens | Validacao: {len(val_data)} tokens")

block_size = 32   # quantos tokens de contexto o modelo ve por vez
batch_size = 16   # quantas sequencias processamos em paralelo

def get_batch(split):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])  # y e x deslocado 1 posicao: o alvo de cada token e o proximo
    return x, y

xb, yb = get_batch("train")
print("Shape de um batch:", xb.shape)  # (batch_size, block_size)


## 3. Self-attention: a peca central

Uma unica "cabeca" de atencao faz o seguinte para cada token da sequencia:

1. Projeta o embedding do token em tres vetores: **Query** (o que estou procurando), **Key** (o que eu ofereco) e **Value** (o conteudo que carrego)
2. Calcula a similaridade entre a Query de cada token e a Key de todos os outros (`Q @ K.T`) — isso da os "scores" de atencao
3. Aplica uma **mascara causal**: zera (com `-inf`) os scores de tokens futuros, para que o modelo nunca "veja" o que ainda nao deveria saber
4. Aplica `softmax` para transformar os scores em pesos que somam 1
5. Usa esses pesos para fazer uma media ponderada dos Values de todos os tokens

O `/ sqrt(d_k)` existe para manter os valores em uma escala estavel antes do softmax (sem isso, o softmax fica "saturado" e os gradientes desaparecem).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Head(nn.Module):
    """Uma unica cabeca de self-attention causal."""
    def __init__(self, n_embd, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        # mascara causal: matriz triangular inferior (1 = pode ver, 0 = nao pode ver)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape  # batch, tempo (sequencia), canais (n_embd)
        k = self.key(x)    # (B, T, head_size)
        q = self.query(x)  # (B, T, head_size)
        v = self.value(x)  # (B, T, head_size)

        # scores de atencao: quanto cada token deve "prestar atencao" em cada outro
        wei = q @ k.transpose(-2, -1) * (k.shape[-1] ** -0.5)   # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))  # mascara causal
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)

        out = wei @ v   # (B, T, head_size) - media ponderada dos Values
        return out


## 4. Multi-head attention

Em vez de uma unica cabeca de atencao, usamos varias em paralelo, cada uma podendo aprender a focar em um tipo diferente de relacao entre tokens (uma cabeca pode aprender a olhar para o token anterior, outra para concordancia de genero, outra para dependencias de longo alcance, etc). No final, concatenamos as saidas de todas as cabecas e projetamos de volta ao tamanho original.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([
            Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)
        ])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # concatena as cabecas
        out = self.dropout(self.proj(out))
        return out


## 5. Feed-forward e bloco Transformer completo

O feed-forward e uma rede simples (linear -> ReLU -> linear) aplicada a cada posicao independentemente — e onde o modelo faz "processamento" adicional depois de coletar informacao via atencao.

O bloco combina atencao + feed-forward com **conexoes residuais** (`x = x + ...`) e **layer normalization**. As conexoes residuais sao essenciais para treinar redes profundas: elas dao ao gradiente um "atalho" direto para fluir de volta sem se degradar (o mesmo problema que backpropagation enfrenta em redes muito profundas, visto na Fase 1).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Um bloco Transformer: atencao + feed-forward, cada um com residual + layer norm."""
    def __init__(self, n_embd, n_head, block_size, dropout=0.1):
        super().__init__()
        self.sa = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ffwd = FeedForward(n_embd, dropout)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # residual em torno da atencao
        x = x + self.ffwd(self.ln2(x))   # residual em torno do feed-forward
        return x


## 6. O modelo GPT completo

Junta tudo:
- **Embedding de token**: transforma cada indice de caractere em um vetor denso
- **Embedding posicional**: soma informacao de *posicao*, ja que a atencao sozinha nao tem nocao de ordem
- **N blocos empilhados**: cada um refinando a representacao
- **Layer norm final + camada linear**: projeta de volta para o tamanho do vocabulario, gerando os "logits" (pontuacao bruta para cada caractere possivel ser o proximo)

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)                              # (B, T, n_embd)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))  # (T, n_embd)
        x = tok_emb + pos_emb        # soma embedding de conteudo + embedding de posicao
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)     # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B * T, C), targets.view(B * T))

        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Gera texto autoregressivamente: preve um token, adiciona a sequencia, repete."""
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]   # trunca ao tamanho maximo de contexto
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature   # so o ultimo passo de tempo importa
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # amostra da distribuicao
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


## 7. Instanciar e treinar o modelo

In [ ]:
torch.manual_seed(42)

n_embd = 64     # dimensao dos embeddings
n_head = 4      # numero de cabecas de atencao
n_layer = 4     # numero de blocos Transformer empilhados
dropout = 0.1

model = MiniGPT(vocab_size, n_embd, n_head, n_layer, block_size, dropout)
n_params = sum(p.numel() for p in model.parameters())
print(f"Modelo com {n_params:,} parametros")

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)


In [ ]:
@torch.no_grad()
def estimate_loss(eval_iters=20):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            _, loss = model(xb, yb)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out


max_iters = 2000
eval_interval = 200

for it in range(max_iters):
    if it % eval_interval == 0 or it == max_iters - 1:
        losses = estimate_loss()
        print(f"passo {it:4d} | loss treino {losses['train']:.4f} | loss val {losses['val']:.4f}")

    xb, yb = get_batch("train")
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()


## 8. Gerar texto

Comecamos a geracao com um unico token (quebra de linha, indice 0 do contexto) e deixamos o modelo gerar caractere por caractere, sempre olhando para tras (nunca para frente, por causa da mascara causal).

Com um corpus tao pequeno, nao espere coerencia — o objetivo aqui e ver o modelo aprender **padroes locais** (silabas, espacamento, pontuacao) do texto de treino.

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long)  # comeca com um token "vazio"
generated = model.generate(context, max_new_tokens=300, temperature=0.8)
print(decode(generated[0].tolist()))


## Notas finais e experimentos sugeridos

- **Corpus maior**: troque o texto de exemplo por um arquivo `.txt` de pelo menos algumas dezenas de KB (ex: um livro em dominio publico, suas proprias anotacoes) — com um corpus tao curto quanto o do exemplo, o modelo tende a decorar em vez de generalizar
- **`temperature`**: teste `0.2` (mais conservador, repetitivo) vs `1.2` (mais aleatorio, "criativo" mas menos coerente) na geracao
- **Escala do modelo**: aumente `n_embd`, `n_head` e `n_layer` e observe o efeito no `loss` de validacao e no tempo de treino
- **Tokenizacao BPE**: troque a tokenizacao por caractere por BPE (veja o video "Let's build the GPT Tokenizer" do Karpathy) e compare a qualidade do texto gerado com o mesmo orcamento de treino
- **GPU**: se tiver GPU disponivel (`torch.cuda.is_available()`), mova o modelo e os batches para `device="cuda"` — o treino fica ordens de magnitude mais rapido, o que permite corpora e modelos bem maiores
